In [8]:
!pip install autogluon

In [9]:
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split


df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df.drop('customerID', axis=1, inplace=True)

# Replace ' ' in TotalCharges with NaN and convert to float
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
#If a value cannot be converted to a number, it is replaced with NaN (missing value).

# Fill missing values (optional, AutoGluon can handle it too)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1


In [ ]:

train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

label = 'Churn'

# Custom metric: Balanced Accuracy
predictor = TabularPredictor(
    label=label, # Target column → Churn
    eval_metric='balanced_accuracy', #Balanced accuracy is useful when dataset is imbalanced.
    problem_type='binary', #problem_type='binary'

#Because churn has two classes:

#Yes

#No
    path='AutoML_Churn_Model'       #Directory where AutoML saves:

#trained models

#logs

#leaderboard

#feature importance
).fit(
    train_data=train_data,
    time_limit=600,  # 10 minutes max time

    # time_limit=600

#Training runs maximum 10 minutes.

#AutoML tries many models until time expires.
    presets='best_quality',  # best ensemble models
    num_bag_folds=5,  # bagging for robustness
    #Train model 5 times on different data splits

#Benefits:

#reduces overfitting

#improves stability

#Example:

#Dataset split into 5 folds.
    num_stack_levels=2,  # stacking for meta-learning
    hyperparameters={
        'GBM': {},  # LightGBM
        'CAT': {'iterations': 1000},
        'XGB': {'n_estimators': 1000},
        'NN_TORCH': {},
        'RF': {},
        'XT': {},
        'KNN': {},
        # 'custom': ['GBM', 'CAT', 'XGB'],
    }
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       9.45 GB / 12.67 GB (74.6%)
Disk Space Avail:   63.96 GB / 112.64 GB (56.8%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequen

Level 1 models
GBM
XGBoost
RandomForest
NeuralNet

        ↓

Level 2 meta model
(learns from predictions)

In [ ]:

leaderboard = predictor.leaderboard(test_data, silent=True)
print("\n Leaderboard:\n", leaderboard)

In [ ]:

print("\n🔎 Feature Importance:")
print(predictor.feature_importance(test_data))

In [ ]:

preds = predictor.predict(test_data.drop(columns=[label]))
probs = predictor.predict_proba(test_data.drop(columns=[label]))[1]

print("\n Metrics:")
performance = predictor.evaluate(test_data)
print(performance)

In [ ]:

predictor.save()  # Save model
loaded_predictor = TabularPredictor.load("AutoML_Churn_Model")